In [23]:
import pandas as pd
import numpy as np
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# Central government debt interest by source, £ billion. Source: OBR EFO Oct 2026, detailed forecast tables: aggregates, Table 6.16.
xl = "../Data/efo-october-2026-detailed-forecast-tables-aggregates.xlsx"   # adjust path
raw = pd.read_excel(xl, sheet_name="6.16", header=None)

# year headers on the row containing "2026-27"; rows picked by label (first "Existing stock" / "New stock" are the conventional gilts)
hdr = raw[raw.apply(lambda r: r.astype(str).str.contains("2026-27").any(), axis=1)].index[0]
years = {c: str(v) for c, v in raw.loc[hdr].items() if str(v)[:2] == "20" and "-" in str(v)}
lab = raw[1].astype(str).str.strip()
row = lambda text, n=0: raw[lab.str.startswith(text)].iloc[n]
existing, new, total = row("Existing stock", 0), row("New stock", 0), row("Total CG debt interest")

d = pd.DataFrame({"fy": list(years.values()),
                  "existing": [float(existing[c]) for c in years], "new": [float(new[c]) for c in years],
                  "total": [float(total[c]) for c in years]})
d = d[d["fy"] != "2025-26"].reset_index(drop=True)          # the existing/new split starts in the first forecast year
d["other"] = d["total"] - d["existing"] - d["new"]          # index-linked gilts, NS&I, APF gilts, other

# sanity check against the EFO: debt interest of £124bn in 2026-27 and £160bn in 2031-32
assert abs(d.loc[0, "total"] - 124) < 1 and abs(d.loc[len(d) - 1, "total"] - 160) < 1

EXISTING, NEW, OTHER = "Conventional gilts already issued", "Conventional gilts yet to be issued", "All other debt interest"
long = d.melt(id_vars="fy", value_vars=["other", "existing", "new"], var_name="key", value_name="value")
long["segment"] = long["key"].map({"existing": EXISTING, "new": NEW, "other": OTHER})
long["stack"] = long["key"].map({"other": 0, "existing": 1, "new": 2})            # bottom to top: new gilts sit on top

# key order = stack order top to bottom. Colours are explicit so the legend works: new debt = ecostyles red, old debt = greys
segments = [NEW, EXISTING, OTHER]
scale = alt.Scale(domain=segments, range=["#E6224B", "#8A8DA6", "#D5D7E3"])
colour = alt.Color("segment:N", scale=scale,
                   legend=alt.Legend(orient="top", title=None, symbolType="square", symbolSize=140, labelLimit=420,
                                     labelFontSize=12, columnPadding=24))

x = alt.X("fy:N", title=None, axis=alt.Axis(labelAngle=0))
y = alt.Y("value:Q", title=None, scale=alt.Scale(domain=[0, 165], nice=False),
          axis=alt.Axis(values=[0, 50, 100, 150], labelExpr="datum.value == 0 ? '0' : '£' + datum.value + 'bn'"))

bars = alt.Chart(long).mark_bar(size=58).encode(x=x, y=y, color=colour, order=alt.Order("stack:Q"))

# hover: shaded column, tooltip lists every segment for that year
hover = alt.selection_point(fields=["fy"], on="mouseover", empty=False, clear="mouseout")
tips = [alt.Tooltip("fy:N", title="Fiscal year")] + [
    alt.Tooltip(f"{k}:Q", format=",.1f", title=f"{t} (£bn)") for k, t in
    [("new", NEW), ("existing", EXISTING), ("other", OTHER), ("total", "Total")]]
hit = alt.Chart(d).mark_bar(size=110, color="#676A86").encode(
    x="fy:N", opacity=alt.condition(hover, alt.value(0.08), alt.value(0)), tooltip=tips).add_params(hover)

caption = alt.Title(
    text="Source: OBR, Economic and fiscal outlook – detailed forecast tables: aggregates (Table 6.16), October 2026",
    subtitle=["Central government debt interest (net of APF), £ billion. Gilts already issued get cheaper; gilts yet to be issued, including refinancing, drive the rise."],
    orient="bottom", anchor="start", fontSize=11, subtitleFontSize=10,
    color="#676A86", subtitleColor="#676A86", dy=12)

chart = (hit + bars).properties(
    width=700, height=270).configure(
    background="white", font="Circular Std, Helvetica Neue, Helvetica, Arial, sans-serif").configure_view(
    fill="transparent", stroke="transparent")

styles.save(chart, path=".", name="debt_interest_sources", svg=True)
chart.save("debt_interest_sources.png", scale_factor=2.0)
chart

alt.LayerChart(...)